# **Prompts Template**


A prompt template is a structured framework that allows for dynamic generation of prompts based on predefined patterns and placeholders. It typically includes fixed text and variables that can be filled with specific values at runtime. Prompt templates are useful when generating multiple prompts with similar structures but varying content or style.


Let's see an usecase how you can define a prompt template where you can configure values as placeholders and execute and pass it as a input to LLM


In [26]:
%%capture
# update or install the necessary libraries
%pip install anthropic python-dotenv

In [27]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# Load environment variables from .env file
load_dotenv()

client = Anthropic()
model = "claude-haiku-5-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    # Haiku 5.5 thinks by default, so content[0] can be a thinking block with no .text.
    # Join the text blocks instead of reading content[0].
    return "".join(block.text for block in message.content if block.type == "text")

In [28]:
# Technique: Prompting (see 001_prompting.ipynb) - a direct, single-turn
# question sent to Claude via add_user_message + chat()
messages = []
add_user_message(messages, """
Today is Monday, tomorrow is Wednesday.

What is wrong with that statement?
""")

response = chat(messages)
response

'Tomorrow would be **Tuesday**, not Wednesday. If today is Monday, tomorrow is the next day in the week, which is Tuesday.'

### Prompt Template Flow

The same template string is reused for every request — only the `variables` change, so `render()` produces a different prompt each time without rewriting the wording.

![Prompt Template Flow](flowdiagram.png)


## **Prompt template**

A prompt template is a fixed template string with placeholders that gets filled in with different values before being sent to Claude.

* **Template String**: A template string is defined, which includes placeholders for `style` and `text`. This template will be used to create dynamic prompts.
* **`render` helper**: A small helper function fills in the placeholders in the template string with real values.
* **Input Variables**: The placeholders found in the template (via a regex) show what needs to be filled in before the prompt can be sent to Claude.

In [29]:
# Technique: Clear and Direct - states exactly
# what to do ("Translate...into a style that is {style}") and uses triple
# backticks to unambiguously delimit the input text
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [ ]:
import re

def render(template_string, variables):
    """Fill in {placeholder} values in a template string with the given variables"""
    result = template_string
    for key, value in variables.items():
        result = result.replace("{" + key + "}", str(value))
    return result


def input_variables(template_string):
    """Return the list of {placeholder} names found in a template string"""
    return re.findall(r"{([^{}]+)}", template_string)

In [31]:
input_variables(template_string)

['style', 'text']

## **Formatting the Employee's Email**

* **Employee Style**: A style specification for translating text is defined (British English in a calm and respectful tone).
* **Employee Email**: A sample French email requesting vacation is provided.
* **Render Prompt**: The `render` helper fills in the template with the specified style and text.
* **Model Response**: The rendered prompt is sent to Claude via `chat`, and the response is printed.

In [32]:
employee_style = """British English \
in a calm and respectful tone
"""

In [33]:
# french language letter for vacation request
employee_email = """
Objet : Demande de Congé

Cher [Nom du Responsable],

Je m'appelle John et je travaille au sein de la société XYZ.
J'aimerais solliciter une demande de congé pour partir en vacances.
Serait-il possible de discuter des dates qui conviendraient le mieux pour l'équipe et l'entreprise?

Je vous remercie par avance pour votre compréhension et j'attends votre retour.

Cordialement,
John
"""

In [34]:
employee_prompt = render(
    template_string,
    {"style": employee_style, "text": employee_email},
)

employee_messages = []
add_user_message(employee_messages, employee_prompt)

In [35]:
employee_prompt

"Translate the text that is delimited by triple backticks into a style that is British English in a calm and respectful tone\n. text: ```\nObjet : Demande de Congé\n\nCher [Nom du Responsable],\n\nJe m'appelle John et je travaille au sein de la société XYZ.\nJ'aimerais solliciter une demande de congé pour partir en vacances.\nSerait-il possible de discuter des dates qui conviendraient le mieux pour l'équipe et l'entreprise?\n\nJe vous remercie par avance pour votre compréhension et j'attends votre retour.\n\nCordialement,\nJohn\n```\n"

In [36]:
employee_response = chat(employee_messages)

In [37]:
employee_response

"**Subject:** Leave Request\n\nDear [Manager's Name],\n\nMy name is John, and I work at XYZ.\n\nI would like to request leave to go on holiday. Would it be possible to discuss the dates that would best suit both the team and the company?\n\nThank you in advance for your understanding. I look forward to hearing from you.\n\nKind regards,\nJohn"

## **Formatting the Manager's Reply**

* **Manager Reply**: A sample reply from a manager is provided in English.
* **Manager Style**: The style for translating the manager's reply is defined (a polite tone that speaks in French).
* **Render Prompt**: The `render` helper is used again to fill the template with the specified style and text.
* **Model Response**: The rendered prompt is sent to Claude via `chat`, and the response is printed.

In [38]:
manager_reply = """
Subject: Re: Demande de Congé

Hi John,

Thank you for reaching out. I've reviewed your request for vacation leave.\
Please provide the specific dates you'd like to take off, so we can ensure proper coverage during your absence.

Looking forward to your response.

Best regards,
[Manager's Name]
"""

In [39]:
manager_style = """\
a polite tone \
that speaks in French\
"""

In [40]:
manager_prompt = render(
    template_string,
    {"style": manager_style, "text": manager_reply},
)

manager_messages = []
add_user_message(manager_messages, manager_prompt)

In [41]:
manager_prompt

"Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in French. text: ```\nSubject: Re: Demande de Congé\n\nHi John,\n\nThank you for reaching out. I've reviewed your request for vacation leave.Please provide the specific dates you'd like to take off, so we can ensure proper coverage during your absence.\n\nLooking forward to your response.\n\nBest regards,\n[Manager's Name]\n```\n"

In [42]:
manager_response = chat(manager_messages)

In [43]:
manager_response

"Objet : Re: Demande de Congé\n\nBonjour John,\n\nJe vous remercie de m'avoir contacté. J'ai bien examiné votre demande de congés. Pourriez-vous me communiquer les dates précises de votre absence, afin que nous puissions assurer une couverture adéquate pendant cette période ?\n\nDans l'attente de votre réponse, je vous prie d'agréer mes salutations distinguées.\n\nCordialement,\n[Nom du responsable]"